In [4]:
# uc_fullpage_lazy_gui.py
import time
import base64
import undetected_chromedriver as uc
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

# ====== ここだけ差し替え ======
TARGET_URL = "https://www.pscube.jp/dedamajyoho-P-townDMMpachi/c745773/cgi-bin/nc-v06-001.php?cd_dai=35"   # ←撮りたいURL

# ====== あなたの open_browser をそのまま使用 ======
def open_browser():
    chrome_options = uc.ChromeOptions()
    chrome_options.add_argument("--incognito")
    chrome_options.add_argument("--disk-cache-size=0")
    chrome_options.add_argument("--disable-blink-features=AutomationControlled")
    chrome_options.add_argument("--disable-gpu")
    chrome_options.add_argument("--disable-extensions")
    chrome_options.add_argument("--disable-application-cache")
    chrome_options.add_argument("--disable-cache")
    chrome_options.add_argument("--no-sandbox")
    chrome_options.add_argument("--disable-dev-shm-usage")
    driver = uc.Chrome(version_main=138, options=chrome_options)  # Chrome 138 を想定
    driver.set_window_size(600, 700)  # ビューポートはCDPキャプチャには依存しない
    return driver

# ====== ページ固有：必要なら追記（「もっと見る」など）======
CLICK_MORE_SELECTORS = [
    "#tblHISTm",
    ".load-more",
    "button[aria-label='もっと見る']",
]

def click_possible_buttons(driver):
    for sel in CLICK_MORE_SELECTORS:
        try:
            btns = driver.find_elements(By.CSS_SELECTOR, sel)
        except Exception:
            btns = []
        for b in btns[:5]:
            try:
                driver.execute_script("arguments[0].scrollIntoView({block:'center'});", b)
                time.sleep(0.1)
                driver.execute_script("arguments[0].click();", b)
                time.sleep(0.3)
            except Exception:
                pass

def force_eager_images(driver):
    # lazy画像をできるだけ即時読込に切り替え
    driver.execute_script("""
        (function(){
            const imgs = document.querySelectorAll('img');
            for (const img of imgs) {
                try {
                    if ('loading' in img) img.loading = 'eager';
                    const lazy = img.getAttribute('data-src') || img.getAttribute('data-lazy-src');
                    if (lazy && (!img.src || img.src.trim() === '')) img.src = lazy;
                } catch(e){}
            }
        })();
    """)

def auto_scroll_to_bottom(driver, max_time=45, step=900, pause=0.4, idle_wait=1.2):
    """
    LazyLoad対策：高さが伸びなくなるまでステップスクロール＋ボタン連打＋画像強制読込。
    """
    start = time.time()
    stable_rounds = 0
    prev_h = 0

    while time.time() - start < max_time:
        # 途中で「もっと見る」っぽいのを押す
        click_possible_buttons(driver)

        # 現在の高さ
        h = driver.execute_script("return Math.max(document.body.scrollHeight, document.documentElement.scrollHeight)")
        if h <= 0:
            break

        # ステップで下まで
        y = driver.execute_script("return window.pageYOffset || document.documentElement.scrollTop || document.body.scrollTop || 0")
        while y + step < h:
            y += step
            driver.execute_script(f"window.scrollTo(0, {y});")
            time.sleep(pause)

        # 最下部へ
        driver.execute_script(f"window.scrollTo(0, {h});")
        time.sleep(pause)

        # 画像を eager に
        force_eager_images(driver)

        # 高さが伸びてるか？
        new_h = driver.execute_script("return Math.max(document.body.scrollHeight, document.documentElement.scrollHeight)")
        if new_h <= h + 5 and abs(new_h - prev_h) <= 5:
            stable_rounds += 1
        else:
            stable_rounds = 0
        prev_h = new_h

        if stable_rounds >= 2:
            break

    time.sleep(idle_wait)  # 最後に少し静寂待ち

def fullpage_screenshot(driver, out_path: str):
    """
    CDPでフルページを1枚に。GUIでもOK。超縦長(~131072px超)は切れる可能性あり。
    """
    metrics = driver.execute_cdp_cmd("Page.getLayoutMetrics", {})
    content = metrics.get("contentSize", {})
    width = float(content.get("width", 0.0)) or float(driver.execute_script("return window.innerWidth || 1200"))
    height = float(content.get("height", 0.0)) or float(driver.execute_script("return document.body.scrollHeight || 2000"))

    # 高DPI対策（失敗したら1.0）
    try:
        dpr = driver.execute_cdp_cmd("Emulation.getMetrics", {}).get("deviceScaleFactor", 1.0)
    except Exception:
        dpr = 1.0

    result = driver.execute_cdp_cmd(
        "Page.captureScreenshot",
        {
            "format": "png",
            "fromSurface": True,
            "captureBeyondViewport": True,
            "clip": {
                "x": 0.0,
                "y": 0.0,
                "width": width,
                "height": height,
                "scale": float(dpr) if dpr else 1.0,
            },
        },
    )
    with open(out_path, "wb") as f:
        f.write(base64.b64decode(result["data"]))

if __name__ == "__main__":
    driver = open_browser()
    try:
        driver.get(TARGET_URL)

        # body出現まで軽く待つ（必要なら任意の要素に変更）
        try:
            WebDriverWait(driver, 20).until(EC.presence_of_element_located((By.TAG_NAME, "body")))
        except Exception:
            pass

        # LazyLoadをできるだけ読み込ませる
        auto_scroll_to_bottom(driver, max_time=45, step=900, pause=0.4, idle_wait=1.2)

        # フルページ一発保存
        out = "fullpage_gui_lazy.png"
        fullpage_screenshot(driver, out)
        print(f"✅ {out} を保存しました")
    finally:
        try:
            driver.quit()
        except Exception:
            pass


✅ fullpage_gui_lazy.png を保存しました
